# Viscosities

In [ ]:
out_path = None

In [ ]:
from pathlib import Path

viscosity_files = [
    Path("../../datasets/v1/viscosities/G50.csv"),
    Path("../../datasets/v1/viscosities/G45.csv"),
    Path("../../datasets/v1/viscosities/G40.csv"),
    Path("../../datasets/v1/viscosities/G40IPA.csv"),
]
pv_files = [
    Path("../../datasets/v1/process_variables/mean_profiles/dataset1.csv"),
    Path("../../datasets/v1/process_variables/mean_profiles/dataset2.csv"),
    Path("../../datasets/v1/process_variables/mean_profiles/dataset3.csv"),
    Path("../../datasets/v1/process_variables/mean_profiles/dataset4.csv"),
    Path("../../datasets/v1/process_variables/mean_profiles/dataset5.csv"),
]

In [ ]:
from itertools import cycle

import matplotlib.pyplot as plt
import pandas as pd

MARKERS = dict(
    G50="o",
    G45="v",
    G40="^",
    G40IPA="s",
)


pvs = [pd.read_csv(pv_file) for pv_file in pv_files]
process_shear_rates = pd.concat([pv["shear_rate"].dropna() for pv in pvs])
shear_rate_min = process_shear_rates.min()
shear_rate_max = process_shear_rates.max()

fig, ax = plt.subplots()
colors = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

for path, color in zip(viscosity_files, colors):
    df = pd.read_csv(path)
    descending = df["sweep_direction"] == "descending"
    in_process_range = df["shear_rate"].between(
        shear_rate_min, shear_rate_max, inclusive="both"
    )

    # Draw the complete curve faintly first, then redraw the measurements
    # covered by actual process conditions at full opacity.
    ax.loglog(
        df["shear_rate"][descending].values,
        df["viscosity"][descending].values,
        marker=MARKERS[path.stem],
        color=color,
        alpha=0.25,
    )
    ax.loglog(
        df["shear_rate"][descending & in_process_range].values,
        df["viscosity"][descending & in_process_range].values,
        marker=MARKERS[path.stem],
        color=color,
        label=path.stem,
    )
ax.set_xlabel("Shear Rate (1/s)")
ax.set_ylabel("Viscosity (Pa·s)")
ax.legend()

if out_path is not None:
    plt.savefig(out_path)
else:
    fig.show()